## 1. Setup and configuration

In [ ]:
!pip -q install kagglehub

import os, glob, random, math, time, warnings
import numpy as np, pandas as pd
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
AMP = device == "cuda"

# ---------------- configuration ----------------
SEED = 42
S = 256
BATCH = 16
EPOCHS = 50
LR = 1e-2
N_TRAIN, N_VAL, N_TEST = 4500, 900, 1500

LAMBDA_CLS, LAMBDA_CAM, LAMBDA_ACE = 0.2, 0.3, 0.5
M_PRIORS = 5
PRIOR_RES = 64
ACE_REGION = 4
N_BINS = 10


MASK_DIR = None

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
print("device:", device)

In [ ]:
import kagglehub
root = kagglehub.dataset_download("hgunraj/covidxct")
print("dataset at:", root)

def find_file(name):
    for depth in range(4):
        hits = glob.glob(os.path.join(root, *["*"] * depth, name))
        if hits: return hits[0]
    raise FileNotFoundError(name)

def find_dir(name):
    for depth in range(4):
        hits = [h for h in glob.glob(os.path.join(root, *["*"] * depth, name)) if os.path.isdir(h)]
        if hits: return hits[0]
    raise FileNotFoundError(name)

IMG_DIR = find_dir("3A_images")

def read_split(split):
    return pd.read_csv(find_file(f"{split}_COVIDx_CT-3A.txt"), sep=" ", header=None,
                       names=["file", "cls", "x1", "y1", "x2", "y2"])

def subsample(df, n, seed):
    parts = [g.sample(min(len(g), n // 3), random_state=seed) for _, g in df.groupby("cls")]
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

dfs = {"train": subsample(read_split("train"), N_TRAIN, SEED),
       "val":   subsample(read_split("val"),   N_VAL,   SEED),
       "test":  subsample(read_split("test"),  N_TEST,  SEED)}
for k, v in dfs.items():
    print(k, len(v), v.cls.value_counts().sort_index().to_dict())

In [ ]:
def load_one(r):
    im = Image.open(os.path.join(IMG_DIR, r.file)).convert("L")
    W, H = im.size
    img = np.asarray(im.resize((S, S), Image.BILINEAR), dtype=np.uint8)
    if MASK_DIR is not None:
        mp = os.path.join(MASK_DIR, os.path.splitext(r.file)[0] + ".png")
        if os.path.exists(mp):
            m = (np.asarray(Image.open(mp).convert("L").resize((S, S), Image.NEAREST)) > 127).astype(np.uint8)
        else:
            m = np.zeros((S, S), np.uint8)
        c = int(m.any())
    else:
        m = np.zeros((S, S), np.uint8)
        x1, x2 = int(r.x1 * S / W), int(math.ceil(r.x2 * S / W))
        y1, y2 = int(r.y1 * S / H), int(math.ceil(r.y2 * S / H))
        m[max(y1, 0):y2, max(x1, 0):x2] = 1
        c = int(r.cls == 2)       # COVID-19 vs. not
    return img, m, c

def build_arrays(df):
    with ThreadPoolExecutor(8) as ex:
        res = list(tqdm(ex.map(load_one, df.itertuples(index=False)), total=len(df)))
    imgs = torch.from_numpy(np.stack([r[0] for r in res]))
    masks = torch.from_numpy(np.stack([r[1] for r in res]))
    cls = torch.tensor([r[2] for r in res])
    return imgs, masks, cls

class SegDS(Dataset):
    def __init__(self, arrays, train=False):
        self.imgs, self.masks, self.cls = arrays
        self.train = train
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        x = self.imgs[i].float().div(255.)[None]
        y = self.masks[i].float()[None]
        if self.train:
            if random.random() < 0.5: x, y = x.flip(-1), y.flip(-1)
            x = x * (1 + 0.2 * (random.random() - 0.5)) + 0.1 * (random.random() - 0.5)
        x = (x - x.mean()) / (x.std() + 1e-6)          # per-image z-score (nnU-Net style CT/MR normalisation)
        return x, y, self.cls[i].float(), i

arrs = {k: build_arrays(v) for k, v in dfs.items()}
train_ds, val_ds, test_ds = (SegDS(arrs["train"], True), SegDS(arrs["val"]), SegDS(arrs["test"]))
def mk_loader(ds, shuffle): return DataLoader(ds, BATCH, shuffle=shuffle, num_workers=2, drop_last=shuffle, pin_memory=True)
train_loader, val_loader, test_loader = mk_loader(train_ds, True), mk_loader(val_ds, False), mk_loader(test_ds, False)
print("positive image-level targets (train/val/test):", [int(arrs[k][2].sum()) for k in ("train", "val", "test")])

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        def cbl(i, o, s): return [nn.Conv2d(i, o, 3, s, 1, bias=False), nn.InstanceNorm2d(o, affine=True), nn.LeakyReLU(0.01, inplace=True)]
        self.net = nn.Sequential(*cbl(cin, cout, stride), *cbl(cout, cout, 1))
    def forward(self, x): return self.net(x)

class NNUNet2D(nn.Module):
    def __init__(self, use_cls=False, use_cam=False, feats=(32, 64, 128, 256, 320, 320)):
        super().__init__()
        assert use_cls or not use_cam, "CAM needs the classification head"
        self.use_cls, self.use_cam = use_cls, use_cam
        n = len(feats)
        self.enc = nn.ModuleList([ConvBlock(1 if i == 0 else feats[i - 1], feats[i], 1 if i == 0 else 2) for i in range(n)])
        self.ups = nn.ModuleList([nn.ConvTranspose2d(feats[i + 1], feats[i], 2, 2) for i in range(n - 2, -1, -1)])
        self.decs = nn.ModuleList([ConvBlock(2 * feats[i], feats[i]) for i in range(n - 2, -1, -1)])
        if use_cls: self.fc = nn.Linear(feats[-1], 1)
        if use_cam: self.refine = nn.Sequential(nn.Conv2d(feats[0] + 1, feats[0], 3, 1, 1, bias=False),
                                                nn.InstanceNorm2d(feats[0], affine=True), nn.LeakyReLU(0.01, inplace=True))
        self.head = nn.Conv2d(feats[0], 1, 1)

    def forward(self, x):
        size = x.shape[-2:]
        skips = []
        for e in self.enc:
            x = e(x); skips.append(x)
        bott = skips[-1]; d = bott
        for j, (up, dec) in enumerate(zip(self.ups, self.decs)):
            d = dec(torch.cat([up(d), skips[len(skips) - 2 - j]], 1))
        out = {}
        if self.use_cls:
            logit = self.fc(bott.mean((2, 3)))                                   # (B,1)
            cam = F.relu(torch.einsum("bchw,c->bhw", bott, self.fc.weight[0])).unsqueeze(1)
            cam = cam / (cam.amax((2, 3), keepdim=True) + 1e-6)
            cam = F.interpolate(cam, size=size, mode="bilinear", align_corners=False)
            out["cls"], out["cam"] = logit, cam
            if self.use_cam:
                d = self.refine(torch.cat([d, cam * torch.sigmoid(logit).view(-1, 1, 1, 1)], 1))
        out["logits"] = self.head(d)
        return out

# quick shape check
_m = NNUNet2D(True, True).to(device)
with torch.no_grad(): _o = _m(torch.randn(2, 1, S, S, device=device))
print({k: tuple(v.shape) for k, v in _o.items()}, "| params: %.1fM" % (sum(p.numel() for p in _m.parameters()) / 1e6))
del _m, _o

In [ ]:
def dice_loss(logits, y, smooth=1.0):
    p = torch.sigmoid(logits)
    inter = (p * y).sum((1, 2, 3)); den = p.sum((1, 2, 3)) + y.sum((1, 2, 3))
    return (1 - (2 * inter + smooth) / (den + smooth)).mean()

def seg_loss(logits, y):                       # L_seg = 1/2 Dice + 1/2 BCE
    return 0.5 * dice_loss(logits, y) + 0.5 * F.binary_cross_entropy_with_logits(logits, y)

def cam_loss(cam, y, c):                       # align CAM with the mask on images with a positive image-level target
    pos = c > 0.5
    if pos.sum() == 0: return cam.sum() * 0
    return F.binary_cross_entropy(cam[pos].clamp(1e-4, 1 - 1e-4), y[pos])

class PriorBank:
    def __init__(self, masks, res=PRIOR_RES):
        self.res = res
        self.bank = F.adaptive_avg_pool2d(masks[:, None].float(), res).flatten(1).to(device)    # (N, res*res)
        self.norm = F.normalize(self.bank, dim=1)
    @torch.no_grad()
    def retrieve(self, prob, M, exclude=None):
        q = F.adaptive_avg_pool2d(prob, self.res).flatten(1)
        sim = F.normalize(q, dim=1) @ self.norm.T                                              # (B, N)
        if exclude is not None: sim[torch.arange(len(q), device=sim.device), exclude.to(sim.device)] = -1
        w, idx = sim.topk(M, dim=1)
        return w.clamp(0, 1), self.bank[idx]                                                    # (B,M), (B,M,res*res)

def uncertainty(prob, bank, M=M_PRIORS, exclude=None):
    # U = 1/M sum_i w_i |y_bar_i - y*|   -> (B,1,res,res)
    prob = prob.float()
    w, priors = bank.retrieve(prob.detach(), M, exclude)
    q = F.adaptive_avg_pool2d(prob, bank.res).flatten(1)
    U = (w[:, :, None] * (priors - q[:, None]).abs()).sum(1) / M
    return U.view(-1, 1, bank.res, bank.res)

def ace_loss(U, prob, y, res=PRIOR_RES, region=ACE_REGION):
    e = (F.adaptive_avg_pool2d(prob, res) - F.adaptive_avg_pool2d(y, res)).abs().detach()
    return (F.avg_pool2d(U, region) - F.avg_pool2d(e, region)).abs().mean()

bank = PriorBank(arrs["train"][1])
print("prior bank:", tuple(bank.bank.shape))

In [ ]:
def autocast(): return torch.autocast("cuda", enabled=AMP)

@torch.no_grad()
def val_dice(model, loader):
    model.eval(); ds = []
    for x, y, c, i in loader:
        x, y = x.to(device), y.to(device)
        with autocast(): out = model(x)
        pb = (torch.sigmoid(out["logits"].float()) > 0.5).float()
        inter = (pb * y).sum((1, 2, 3))
        ds.append((2 * inter + 1) / (pb.sum((1, 2, 3)) + y.sum((1, 2, 3)) + 1))
    return torch.cat(ds).mean().item()

@torch.no_grad()
def evaluate(model, loader, bank, n_vis=0):
    model.eval()
    per = {k: [] for k in ("dice", "iou", "prec", "rec")}
    TP = FP = FN = 0.0
    nb = N_BINS
    cnt = torch.zeros(2, nb, device=device, dtype=torch.float64); sconf = torch.zeros_like(cnt); sacc = torch.zeros_like(cnt)
    g = torch.zeros(6, dtype=torch.float64, device=device)      # n, sum u, sum e, sum u^2, sum e^2, sum ue
    uerr = uok = nerr = nok = 0.0
    pcorr, img_u, img_e, vis = [], [], [], []
    for x, y, c, idx in loader:
        x, y = x.to(device), y.to(device)
        with autocast(): out = model(x)
        prob = torch.sigmoid(out["logits"].float()); pb = (prob > 0.5).float()
        inter = (pb * y).sum((1, 2, 3)); ps = pb.sum((1, 2, 3)); ys = y.sum((1, 2, 3))
        dice = (2 * inter + 1) / (ps + ys + 1)
        per["dice"].append(dice); per["iou"].append((inter + 1) / (ps + ys - inter + 1))
        per["prec"].append((inter + 1) / (ps + 1)); per["rec"].append((inter + 1) / (ys + 1))
        TP += inter.sum().item(); FP += (ps - inter).sum().item(); FN += (ys - inter).sum().item()
        # calibration bins (class 0 = foreground, class 1 = background)
        for k, (conf, ind) in enumerate([(prob, y), (1 - prob, 1 - y)]):
            b = (conf * nb).long().clamp(0, nb - 1).flatten()
            cnt[k] += torch.bincount(b, minlength=nb).double()
            sconf[k] += torch.bincount(b, weights=conf.flatten().double(), minlength=nb)
            sacc[k] += torch.bincount(b, weights=ind.flatten().double(), minlength=nb)
        # uncertainty vs. error
        U = uncertainty(prob, bank)
        err = F.adaptive_avg_pool2d((pb != y).float(), bank.res)
        u, e = U.flatten(1).double(), err.flatten(1).double()
        uc, ec = u - u.mean(1, keepdim=True), e - e.mean(1, keepdim=True)
        den = uc.norm(dim=1) * ec.norm(dim=1); ok = den > 1e-8
        pcorr += ((uc * ec).sum(1)[ok] / den[ok]).tolist()
        has = e > 0
        uerr += u[has].sum().item(); nerr += has.sum().item(); uok += u[~has].sum().item(); nok += (~has).sum().item()
        g += torch.stack([torch.tensor(float(u.numel()), device=device, dtype=torch.float64), u.sum(), e.sum(), (u * u).sum(), (e * e).sum(), (u * e).sum()])
        img_u += u.mean(1).tolist(); img_e += (1 - dice).tolist()
        if len(vis) < n_vis:
            cam = out.get("cam", None)
            for j in range(len(x)):
                if len(vis) >= n_vis: break
                vis.append(dict(x=x[j, 0].cpu(), y=y[j, 0].cpu(), p=prob[j, 0].cpu(),
                                cam=None if cam is None else cam[j, 0].float().cpu(),
                                U=F.interpolate(U[j:j + 1], size=x.shape[-2:], mode="bilinear")[0, 0].cpu()))
    n, su, se, suu, see, sue = g.tolist()
    cov = sue / n - (su / n) * (se / n)
    gcorr = cov / (math.sqrt(max(suu / n - (su / n) ** 2, 1e-12)) * math.sqrt(max(see / n - (se / n) ** 2, 1e-12)))
    valid = cnt > 0
    o = sacc / cnt.clamp(min=1); ex = sconf / cnt.clamp(min=1)
    ace = (((o - ex).abs() * valid).sum() / valid.sum()).item()
    m = {k: torch.cat(v).mean().item() for k, v in per.items()}
    res = {"Dice": m["dice"], "IoU": m["iou"], "Precision": m["prec"], "Recall": m["rec"],
           "F1": 2 * TP / (2 * TP + FP + FN + 1e-9),
           "ACE": ace,
           "U_mean": su / n, "U_on_error": uerr / max(nerr, 1), "U_on_correct": uok / max(nok, 1),
           "corr_U_err_pixel": gcorr, "corr_U_err_per_image": float(np.mean(pcorr)) if pcorr else float("nan"),
           "corr_meanU_vs_(1-Dice)": float(np.corrcoef(img_u, img_e)[0, 1])}
    return res, vis

In [ ]:
CKPT_DIR = "/content/ckpt"; os.makedirs(CKPT_DIR, exist_ok=True)

VARIANTS = {   # name: (use_CH, use_CAM, use_ACE)
    "nnU-Net":             (False, False, False),
    "nnU-Net+CH":          (True,  False, False),
    "nnU-Net+CH+CAM":      (True,  True,  False),
    "nnU-Net+CH+CAM+ACE":  (True,  True,  True),
}

def train_variant(name, use_ch, use_cam, use_ace):
    set_seed(SEED)
    model = NNUNet2D(use_ch, use_cam).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=LR, momentum=0.99, nesterov=True, weight_decay=3e-5)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    best, best_path, hist = -1, f"{CKPT_DIR}/{name}.pt", []
    for ep in range(EPOCHS):
        for gp in opt.param_groups: gp["lr"] = LR * (1 - ep / EPOCHS) ** 0.9          # poly schedule
        model.train(); agg = {"tot": 0, "seg": 0, "cls": 0, "cam": 0, "ace": 0}; nit = 0
        for x, y, c, idx in tqdm(train_loader, desc=f"{name} ep{ep + 1}/{EPOCHS}", leave=False):
            x, y, c = x.to(device), y.to(device), c.to(device)
            with autocast(): out = model(x)
            logits = out["logits"].float(); prob = torch.sigmoid(logits)
            Ls = seg_loss(logits, y); L = Ls; Lc = Lm = La = torch.zeros((), device=device)
            if use_ch:
                Lc = F.binary_cross_entropy_with_logits(out["cls"].float().squeeze(1), c); L = L + LAMBDA_CLS * Lc
            if use_cam:
                Lm = cam_loss(out["cam"].float(), y, c); L = L + LAMBDA_CAM * Lm
            if use_ace:
                U = uncertainty(prob, bank, M_PRIORS, exclude=idx)
                La = ace_loss(U, prob, y); L = L + LAMBDA_ACE * La
            opt.zero_grad(set_to_none=True)
            scaler.scale(L).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 12)
            scaler.step(opt); scaler.update()
            for k, v in zip(agg, (L, Ls, Lc, Lm, La)): agg[k] += float(v)
            nit += 1
        vd = val_dice(model, val_loader)
        hist.append({"epoch": ep + 1, "val_dice": vd, **{k: v / nit for k, v in agg.items()}})
        print(f"[{name}] ep{ep + 1:02d} loss {agg['tot'] / nit:.4f} (seg {agg['seg'] / nit:.3f} cls {agg['cls'] / nit:.3f} "
              f"cam {agg['cam'] / nit:.3f} ace {agg['ace'] / nit:.3f}) | val Dice {vd:.4f}")
        if vd > best:
            best = vd; torch.save(model.state_dict(), best_path)
    model.load_state_dict(torch.load(best_path))          # best-on-VALIDATION checkpoint
    return model, pd.DataFrame(hist)

models, histories, results, vis_store = {}, {}, {}, {}
for name, flags in VARIANTS.items():
    t0 = time.time()
    models[name], histories[name] = train_variant(name, *flags)
    results[name], vis_store[name] = evaluate(models[name], test_loader, bank, n_vis=8)
    print(f"== {name}: done in {(time.time() - t0) / 60:.1f} min | test Dice {results[name]['Dice']:.4f}\n")

In [ ]:
res_df = pd.DataFrame(results).T
seg_cols = ["Dice", "IoU", "Precision", "Recall", "F1"]
unc_cols = ["ACE", "U_mean", "U_on_error", "U_on_correct", "corr_U_err_pixel", "corr_U_err_per_image", "corr_meanU_vs_(1-Dice)"]
print("Segmentation (test)"); display(res_df[seg_cols].round(4))
print("Calibration / uncertainty (test)"); display(res_df[unc_cols].round(4))
res_df.to_csv("/content/ablation_results.csv")


fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for n, h in histories.items():
    ax[0].plot(h.epoch, h.seg, label=n); ax[1].plot(h.epoch, h.val_dice, label=n)
ax[0].set_title("train L_seg"); ax[1].set_title("val Dice"); ax[1].legend(fontsize=7)
for a in ax: a.set_xlabel("epoch")
plt.tight_layout(); plt.show()

In [ ]:
def show_qualitative(name, n=6):
    v = vis_store[name][:n]
    cols = ["Image", "Ground truth", "Prediction", "CAM", "Uncertainty U", "Error |pred-GT|"]
    fig, axs = plt.subplots(len(v), 6, figsize=(14, 2.3 * len(v)))
    axs = np.atleast_2d(axs)
    for r, s in enumerate(v):
        axs[r, 0].imshow(s["x"], cmap="gray")
        axs[r, 1].imshow(s["x"], cmap="gray"); axs[r, 1].imshow(s["y"], cmap="Greens", alpha=0.45)
        axs[r, 2].imshow(s["x"], cmap="gray"); axs[r, 2].imshow((s["p"] > 0.5).float(), cmap="Reds", alpha=0.45)
        axs[r, 3].imshow(s["x"], cmap="gray")
        if s["cam"] is not None: axs[r, 3].imshow(s["cam"], cmap="jet", alpha=0.5, vmin=0, vmax=1)
        im = axs[r, 4].imshow(s["U"], cmap="magma"); plt.colorbar(im, ax=axs[r, 4], fraction=0.046)
        axs[r, 5].imshow((s["p"] > 0.5).float().sub(s["y"]).abs(), cmap="gray", vmin=0, vmax=1)
        for a in axs[r]: a.axis("off")
    for j, t in enumerate(cols): axs[0, j].set_title(t, fontsize=9)
    fig.suptitle(name); plt.tight_layout(); plt.show()

show_qualitative("nnU-Net+CH+CAM+ACE")
